# Renewables Migration: Chapter 1 Proof Engine
### Proving every claim, equation, and number in Chapter 1 of 'The Renewables Migration' by Vincenzo Grimaldi.

This notebook provides a step-by-step verification of the mathematical and economic claims made in Chapter 1.

In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from chapter1_core import Chapter1ProofEngine
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

engine = Chapter1ProofEngine(data_path='../data/book_numbers.csv')
print("Engine Initialized. 2025 Inertia Floor:", engine.get_book_value('inertia_2025_floor'), "GVA*s")

## 1. The 130 GVA·s Cliff
The book claims that in 2025, Continental Europe hit a record low of ~130 GVA·s. Let's visualize the RoCoF sensitivity around this value.

In [ ]:
def plot_rocof(delta_p=1.0):
    h_range = np.linspace(50, 250, 100)
    rocof_vals = [engine.calculate_rocof(delta_p, h) for h in h_range]
    
    plt.figure(figsize=(10, 6))
    plt.plot(h_range, rocof_vals, label='RoCoF (Hz/s)')
    plt.axvline(x=130, color='red', linestyle='--', label='The 130 GVA·s Cliff')
    plt.xlabel('System Inertia (H_sys) [GVA·s]')
    plt.ylabel('RoCoF [Hz/s]')
    plt.title(f'RoCoF Sensitivity (ΔP = {delta_p} GW)')
    plt.legend()
    plt.grid(True)
    plt.show()

interact(plot_rocof, delta_p=FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0));

## 2. The 3D Stability Manifold (Figure 1.2)
Stability is no longer just a function of mass ($H_{sys}$), but an inverse function of protocol latency.

In [ ]:
h_range = np.linspace(0.1, 8, 50)
l_range = np.linspace(0, 100, 50)
H, L = np.meshgrid(h_range, l_range)
Z = np.vectorize(engine.stability_margin_surface)(H, L)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(H, L, Z, cmap='viridis')
ax.set_xlabel('Inertia (H_sys)')
ax.set_ylabel('MCP Latency (ms)')
ax.set_zlabel('Stability Margin')
ax.set_title('Figure 1.2: The 3D Stability Manifold')
plt.show()

## 3. The Protocol Dividend
The book claims a surge to over €3 billion annually in redispatch costs. Let's verify the savings from the MCP-enabled migration.

In [ ]:
bau_2030 = engine.cost_protocol_pivot(2030, 'BAU')
mcp_2030 = engine.cost_protocol_pivot(2030, 'MCP')
dividend = bau_2030 - mcp_2030

print(f"2030 BAU Cost: €{bau_2030:.1f} Billion")
print(f"2030 MCP Cost: €{mcp_2030:.1f} Billion")
print(f"The Protocol Dividend: €{dividend:.1f} Billion saved annually.")